# The Trip Planner

Instead of a digital twin, I have decided to create a trip planner which is more relevant to me given that I have a trip coming up very soon.



In [ ]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
from rich.console import Console
from ddgs import DDGS
from ddgs.exceptions import DDGSException
import json
import os
import requests
import gradio as gr

In [ ]:
# The usual start

load_dotenv(override=True)
azure_api_key = os.getenv("AZURE_API_KEY")
endpoint = os.getenv("AZURE_ENDPOINT")
deployment_name_gpt_5_mini = "gpt-5-mini"
deployment_name_gpt_4_1_nano = "gpt-4.1-nano"

openai = OpenAI(api_key=azure_api_key, base_url=endpoint)

In [ ]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
# push("Check! Check!! Testing Pushover notification from the Trip Planner app.")

In [ ]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
# Some lists!

checklist = []
completed = []

In [ ]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [ ]:
# get_checklist_report()

In [ ]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [ ]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    # Console().print(completion_notes)
    show(completion_notes)
    return get_checklist_report()

In [ ]:
# checklist, completed = [], []

# create_checklist(["Buy groceries", "Finish week 1", "Eat banana", "Read a book"])

In [ ]:
# mark_complete(1, "bought")

In [ ]:
def record_user_details(name, destination, no_of_travelers, days, email="Email not provided", notes="not provided"):
    print(f"Tool called to record a user's trip detail: Name: {name}, Destination: {destination}, Travelers: {no_of_travelers}, Days: {days}, Email: {email}, Notes: {notes}")
    with open("trip_records.txt", "a", encoding="utf-8") as f:
        f.write(f"Name: {name}, Destination: {destination}, Travelers: {no_of_travelers}, Days: {days}, Email: {email}, Notes: {notes}\n")
    return "Trip details recorded"

In [ ]:
def record_trip_details(name, destination, no_of_travelers, days, email="Email not provided", notes="not provided"):
    if int(no_of_travelers) == 1:
        traveling = "alone"
    elif int(no_of_travelers) > 1:
        traveling = "in a group"
    else:
        traveling = "unknown"
    push(f"Recording trip details from {name} with email {email} and notes {notes} traveling {traveling} to {destination} for {days} days.")
    record = record_user_details(name, destination, no_of_travelers, days, email, notes)
    print(record)
    return "OK"

In [ ]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [ ]:
def search_web(query: str, max_results: int = 5) -> dict:
    """Search the web and return result titles, URLs, and snippets."""
    if not query.strip():
        raise ValueError("query must not be empty")
    if isinstance(max_results, bool) or not 1 <= max_results <= 10:
        raise ValueError("max_results must be an integer from 1 to 10")

    try:
        results = DDGS().text(query, max_results=max_results)
    except DDGSException as exc:
        return {"query": query, "results": [], "error": f"Web search returned no usable results or failed: {exc}",}

    return {
        "query": query,
        "results": [
            {
                "title": item.get("title", ""),
                "url": item.get("href", ""),
                "snippet": item.get("body", "")[:3000],
            }
            for item in results
        ],
    }

In [ ]:
# tools json schema

record_trip_details_json = {
    "name": "record_trip_details",
    "description": "Record or update the trip details provided by the user. Use null for unknown fields; do not infer values.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": ["string", "null"], "description": "The name of this user"},
            "destination": {"type": ["string", "null"], "description": "The destination this user is interested in traveling to"},
            "no_of_travelers": {"type": ["integer", "null"], "description": "The number of travelers this user is planning for"},
            "days": {"type": ["integer", "null"], "description": "The number of days this user is planning to travel for"},
            "email": {"type": ["string", "null"], "description": "The email address of this user"},
            "notes": {"type": ["string", "null"], "description": "Any additional info about the conversation that's worth recording to give context"}
        },
        "required": ["name", "destination", "no_of_travelers", "days"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
search_web_json = {
    "name": "search_web",
    "description": (
        "Search the web for current information. Prefer official attraction, restaurant, and transport-provider websites when checking their details or prices. "
        "Return source URLs with the findings."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query. Use site:domain.com to prioritize a specific official website."},
            "max_results": {"type": "integer", "description": "Number of results to return, from 1 to 10.", "default": 5}
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

In [ ]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {"type": "array", "description": "The descriptions of checklist items", "items": {"type": "string"}},
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [ ]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [ ]:
tools = [
            {"type": "function", "function": record_trip_details_json},
            {"type": "function", "function": record_unknown_question_json},
            {"type": "function", "function": search_web_json},
            {"type": "function", "function": create_checklist_json},
            {"type": "function", "function": mark_complete_json}
        ]

In [ ]:
# tools

In [ ]:
"""
for i, tool in enumerate(tools):
    print(i, type(tool))
    print("  function value:", repr(tool.get("function")))
    print("  function value type:", type(tool.get("function")))
"""

In [ ]:
tool_handlers = {
    "record_trip_details": record_trip_details,
    "record_unknown_question": record_unknown_question,
    "search_web": search_web,
    "create_checklist": create_checklist,
    "mark_complete": mark_complete
}

In [ ]:
def handle_tool_calls(tool_calls):
    results = []

    for tool_call in tool_calls:
        name = tool_call.function.name
        print(f"Tool called: {name}", flush=True)

        try:
            tool = tool_handlers[name]
        except KeyError:
            raise ValueError(f"Unknown tool: {name}") from None

        arguments = json.loads(tool_call.function.arguments)
        if not isinstance(arguments, dict):
            raise ValueError(f"Arguments for {name} must be a JSON object")

        result = tool(**arguments)
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id,})

    return results

In [ ]:
system_prompt = """
# Your role

You are a practical trip-planning assistant. Help users plan clear, concise, realistic trips based on their destination, group size, trip length, and preferences.

# Collecting and recording trip details

The trip record has these fields:
- name
- destination
- number_of_travelers
- number_of_days
- email
- additional_notes

When the user provides or changes trip details, call the `record_trip_details` tool with all six fields. Include every field in the tool call: use null for details not yet provided, and an empty string for additional_notes if there are no notes. Do not invent missing details. If details arrive across multiple messages, preserve previously supplied values when updating the record.

Before preparing a personalized itinerary, make sure you know the user's name, destination, number of travelers, and number of days. Ask only for required details that are missing. Email and additional notes are optional; do not require an email to plan the trip.

# Research and planning workflow

Once the required trip details are available, use the checklist tools to manage any multi-step research and itinerary-planning task.

1. Create a concise, ordered checklist of the work needed to produce the trip plan. Include relevant tasks such as checking attraction details, researching local transit routes and fares, checking meal options and costs, organizing activities by day, and verifying the cost totals.
2. Work through the checklist in order. Use the available research tools and prefer reliable, primary sources as described below.
3. Call `mark_complete` to mark a checklist item complete only after you have finished and verified that work. Do not mark unfinished, skipped, or unverified work complete.
4. If new research tasks become necessary, update the checklist before doing them, then mark them complete when finished.
5. Before responding, make sure every applicable checklist item is complete. If something could not be verified or completed, do not mark it complete; explain the limitation clearly in the response.

Use these tools to track the work, not as a substitute for doing it. Do not invent missing trip details to complete the checklist. If a required detail is missing, ask the user for it. For research details that cannot be confirmed, clearly label any grounded estimate or state that the detail could not be verified.

Keep the checklist focused on the current trip-planning request. Do not show it to the user unless they ask for it.

# Research and accuracy

Use current, reliable sources when researching. Prefer primary sources, especially official attraction, transit-agency, restaurant, and transport-provider websites. Use reputable secondary sources only when primary information is unavailable.

Do not invent or imply that you verified prices, schedules, routes, opening hours, or availability when you have not. Label unverified amounts as estimates, and say when information could not be confirmed. Include source links for important practical details and note when information was checked, when possible.

If travel dates are not provided and schedules, prices, or opening hours depend on them, explain that the information may vary by date. Do not claim booking availability unless a source or tool confirms it.

If web search returns an error or no results, do not invent findings or imply that research succeeded. You may retry once with a simpler query; if it still fails, tell the user which details could not be verified and provide no unsupported claims.

# Itinerary and pacing

Once the required trip details are available, provide the itinerary itself in the same response. Do not stop at a list of recommended attractions, and do not ask whether the user wants a day-by-day plan.

Organize the plan under exactly one heading for every trip day, in order:
## Day 1
## Day 2
...
## Day N

N must equal the user's number of days. Do not combine days, skip days, or put the activities in an undated list instead. If the number of days is missing, ask for it rather than inventing a trip length.

Under each day, include:
- **Plan:** One suitable main activity, plus nearby additions only if the day remains comfortable. Include practical timing or pacing when useful.
- **Food:** A relevant meal or restaurant suggestion, with an estimated cost when available.
- **Getting around:** The recommended route or transport mode between that day's activities, with travel time, fare, and ticket details when verified. If walking is practical, say so and give an approximate walking time.
- **Estimated daily cost:** Give a group estimate for the stated number of travelers, and clearly identify exclusions or unverified amounts.

Leave time for meals, transit, and rest. Avoid listing attractions that are not assigned to a day. Do not fill the final day with optional alternatives; recommend a sensible plan for that day too.

After all day headings, include:
## Estimated trip total
Summarize attraction, meal, and local transport costs for the full group and trip. Ensure the arithmetic matches the daily estimates. State important exclusions, such as lodging and flights, and label estimates that could not be verified.

## Sources
Link to the official sources used for important prices, hours, fares, routes, and ticket rules. If a detail could not be verified, say so instead of presenting it as confirmed.

# Response style

Be concise and practical. When the required trip details are available, deliver the complete itinerary directly—do not end by asking whether the user wants an itinerary, restaurant suggestions, or transportation planning. Ask a follow-up only when essential information is missing or a genuine ambiguity prevents a reliable plan.
"""

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=deployment_name_gpt_5_mini, messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model=deployment_name_gpt_5_mini, messages=messages, tools=tools)
    return response.choices[0].message.content

In [ ]:
WELCOME_MESSAGE = f"""
    Hi! I'm Jowie, your trip planning assistant. I can help you plan your trip.

    To get started, please share your name, destination, number of travelers, and number of days.
    You can also include your email (optional) and any useful details, such as your budget, interests, travel dates, or accessibility needs.
"""

chatbot = gr.Chatbot(value=[{"role": "assistant", "content": WELCOME_MESSAGE}])

gr.ChatInterface(fn=chat, chatbot=chatbot).launch(inbrowser=True)